# 05 — SFT training (QLoRA)

Train the first QLoRA adapter on the pinned Qwen3.8-27B base. Run this notebook on Kaggle with a GPU — the cells below skip cleanly on a host without one. Checkpoints are written to the output directory so a session interruption can resume.

11h guard: the run is bounded by `runtime.max_runtime_hours` (default 10h) and `training.max_steps` (600) in the configs. If the wall clock runs out, training stops gracefully with a checkpoint and a `partial_complete` record instead of being killed by Kaggle at 12h — see `src/training/runtime_guard.py`.

Base weights: loaded from Hugging Face in-kernel at the pinned revision and quantized to 4-bit NF4 there (`src/training/model_loader.py`); the kernel needs internet enabled. The notebook is packaged by `scripts/package_kernel.py` into `kernels/sft/`; only this notebook file is uploaded to Kaggle, so the first code cell clones this public repo and runs from the clone, where everything it imports lives.

In [ ]:
# AKARUI_KERNEL_SELF_BOOTSTRAP
# Kaggle kernels receive ONLY this notebook file: `kaggle kernels push`
# uploads the single code_file from kernel-metadata.json, so the src/,
# scripts/, configs/, evaluation/ and data/ files the packager copies
# into kernels/<name>/ never reach the kernel. Self-bootstrap from the
# public repo instead (internet is enabled on the kernel) and run from
# the clone, so the relative paths below resolve. Idempotent: a re-run,
# or a local run from a repo root, skips the clone.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Monibee-Fudgekins/Qwen3.8-27B-akarui"
REPO_DIR = Path("Qwen3.8-27B-akarui")
if (Path("src") / "__init__.py").is_file() and Path("scripts").is_dir():
    print("repo files already present; skipping clone")
elif (REPO_DIR / "src" / "__init__.py").is_file():
    os.chdir(REPO_DIR)
    print("repo root:", Path.cwd())
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"cloning {REPO_URL} ...")
    try:
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True,
        )
    except OSError as exc:
        raise RuntimeError(f"repo clone failed (git not runnable): {exc}") from exc
    if result.returncode != 0:
        raise RuntimeError(
            "repo clone failed; the kernel cannot run without it: "
            + (result.stderr.strip() or result.stdout.strip())
        )
    os.chdir(REPO_DIR)
    print("cloned repo at:", Path.cwd())


In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
print("GPU available:", HAS_GPU)
if not HAS_GPU:
    print("Skipping GPU steps on this host — run this notebook on Kaggle with a GPU.")

In [ ]:
# AKARUI_KERNEL_INSTALL_REQUIREMENTS
# Install the kernel's declared dependencies (requirements-kaggle.txt).
# The Kaggle image ships torch/transformers but NOT bitsandbytes, so a
# 4-bit model load fails with ImportError unless we install it here
# (run #8, 2026-11-21). pip skips packages already satisfied, so this
# is fast when the image already has them. Skipped without a GPU so
# local/no-GPU runs stay light.
import subprocess, sys
if HAS_GPU:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"pip install requirements-kaggle.txt failed (exit {result.returncode})")
    print("requirements-kaggle.txt installed")
else:
    print("No GPU: dependency install skipped.")


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "scripts/prepare_training.py", "--config", "configs/sft.yaml", "--allow-no-gpu"], capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
if HAS_GPU:
    from src.training.trainer_utils import load_config
    from src.training.sft import run_sft
    config = load_config("configs/sft.yaml")
    summary = run_sft(config, config["dataset"]["train_file"], config["output_dir"])
    print(summary)
else:
    print("No GPU: training skipped. On Kaggle, re-run this cell with a GPU attached.")

In [ ]:
from pathlib import Path
out = Path("output/akarui-sft-v0")
print("adapter dir exists:", out.exists())
if out.exists():
    print(sorted(p.name for p in out.iterdir()))